In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#======= Read in the GCMC, TAL Prediction and IAST Prediction data ========
# P-X
df = pd.read_csv('../../../../Datasets/NU-800/Mixture_px.csv')
df2 = pd.read_csv('../../../../IAST_Calculation/NU-800/iast_pred_px.csv')
TAL_px = pd.read_csv('Pred_TAL_seed1.csv')

# # AL
# gf = pd.read_csv('Pred_AL.csv')

In [ ]:
#======= Define Plot Parameters and Plot Data ========
mole_frac = [0.02, 0.06, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.98]

# P-X
row_labels = ["(a)", "(b)", "(c)"]
row_y_positions = [0.915, 0.609, 0.30]
fig, axes = plt.subplots(figsize = (20, 18), nrows=3, ncols=3)

for ax, i in zip(axes.flat, mole_frac):
    # GCMC
    press_gcmc = df[df['Mole Fraction CO2'] == i]['Pressure [Pa]'] / 1e5
    upt_ch_gcmc = df[df['Mole Fraction CO2'] == i]['Uptake_CH4 [mg/gr]']
    upt_co_gcmc = df[df['Mole Fraction CO2'] == i]['Uptake_CO2 [mg/gr]']
    error_ch = df[df['Mole Fraction CO2'] == i]['Error_CH4 [mg/gr]']
    error_ch = 2 * error_ch
    error_co = df[df['Mole Fraction CO2'] == i]['Error_CO2 [mg/gr]']
    error_co = 2 * error_co

    # IAST
    press_iast = df2[df2['Mole Fraction CO2'] == i]['Pressure [Pa]'] / 1e5
    upt_ch_iast = df2[df2['Mole Fraction CO2'] == i]['IAST_CH4_uptake']
    upt_co_iast = df2[df2['Mole Fraction CO2'] == i]['IAST_CO2_uptake']

    # TAL
    press_tal = TAL_px[TAL_px['CO2_mole_fraction'] == i]['X Test']
    upt_ch_tal = TAL_px[TAL_px['CO2_mole_fraction'] == i]['CH4_Pred']
    upt_co_tal = TAL_px[TAL_px['CO2_mole_fraction'] == i]['CO2_Pred']

    # CH4
    ax.plot(press_gcmc, upt_ch_gcmc, marker='o', markersize='6', markeredgecolor='darkmagenta', linestyle=':', linewidth='4', color='purple', label="$CH_{4}$ (GCMC)")
    ax.fill_between(press_gcmc, (upt_ch_gcmc - error_ch), (upt_ch_gcmc + error_ch),color='gray', alpha=0.4)
    ax.plot(press_iast, upt_ch_iast, marker='s', markersize='6', markeredgecolor='mediumvioletred', linestyle=':', linewidth='4', color='pink', label="$CH_{4}$ (IAST)")
    ax.plot(press_tal, upt_ch_tal, marker='p', markersize='6', markeredgecolor='r', linestyle=':', linewidth='4', color='crimson', label="$CH_{4}$ (TAL)")
    # ax.plot(press_al, upt_ch_al, marker='p', markersize='6', markeredgecolor='r', linestyle=':', linewidth='4', color='red', label="$CH_{4}$ (AL)")
    
    # CO2
    ax.plot(press_gcmc, upt_co_gcmc, marker='^', markersize='6', markeredgecolor='b', linestyle=':', linewidth='4', color='steelblue', label="$CO_{2}$ (GCMC)")
    ax.plot(press_iast, upt_co_iast, marker='P', markersize='6', markeredgecolor='g', linestyle=':', linewidth='4', color='mediumseagreen', label="$CO_{2}$ (IAST)")
    ax.plot(press_tal, upt_co_tal, marker='D', markersize='6', markeredgecolor='c', linestyle=':', linewidth='4', color='cyan', label="$CO_{2}$ (TAL)")
    # ax.plot(press_al, upt_co_al, marker='D', markersize='6', markeredgecolor='r', linestyle=':', linewidth='4', color='orange', label="$CO_{2}$ (AL)")
    ax.fill_between(press_gcmc, (upt_co_gcmc - error_co), (upt_co_gcmc + error_co),color='gray', alpha=0.4,label="95% Confidence")
    
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_title("Adsorption Isotherms Comparison at $X_{CO_{2}}$ = " + f'{i}', fontsize = 17)
    ax.set_xlabel("Pressure (bar)", fontsize = 15, fontweight="bold")
    ax.set_ylabel("Adsorption Uptake (mg/g)", fontsize = 15, fontweight="bold")
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontsize(13)
        tick.set_fontweight("bold")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=7, fontsize=17, frameon=False, bbox_to_anchor=(0.5, 0.96))
# Leave room for legend at top and row labels at left
fig.subplots_adjust(left=0.075,right=0.98,bottom=0.055,top=0.88,wspace=0.25,hspace=0.35)
for label, y in zip(row_labels, row_y_positions):
    fig.text(0.025,y,label,fontsize=15,fontweight="bold",ha="left",va="center")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('Adsorption_Isotherm_Comp_px.png', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.savefig('Adsorption_Isotherm_Comp_px.pdf', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.show()

In [ ]:
#======= Generate Residual Contour Plots ========
# Compute residual P-X
residual_ch = df['Uptake_CH4 [mg/gr]'] - TAL_px['CH4_Pred']
residual_ch = np.abs(residual_ch)

residual_co = df['Uptake_CO2 [mg/gr]'] - TAL_px['CO2_Pred']
residual_co = np.abs(residual_co)

vmin = min(residual_co.min(), residual_ch.min())
vmax = max(residual_co.max(), residual_ch.max())

df['P_BAR'] = df['Pressure [Pa]'] / 1e5

mask = np.isfinite(df['Mole fraction CH4']) & np.isfinite(df['P_BAR']) & np.isfinite(residual_ch) & np.isfinite(residual_co)
print(f"Dropping {(~mask).sum()} of {len(mask)} non-finite points")

panel_labels = ["(a)", "(b)"]
fig, ax = plt.subplots(figsize=(11, 5), ncols=2)

contour_ch = ax[0].tricontourf(df['Mole fraction CH4'][mask], df['P_BAR'][mask], residual_ch[mask], levels=30, cmap="YlGnBu", vmin = vmin, vmax = vmax)
ax[0].set_xlabel(r"$\mathbf{CH}_{\mathbf{4}}$ mole fraction", fontsize = 14, fontweight="bold")
ax[0].set_ylabel("Pressure (bar)", fontsize = 14, fontweight="bold")
ax[0].set_title(r"$\mathrm{CH}_{4}$ Prediction Residuals for P-X Space", fontsize = 14)
for tick in ax[0].get_xticklabels() + ax[0].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")

contour_co = ax[1].tricontourf(df['Mole Fraction CO2'][mask], df['P_BAR'][mask], residual_co[mask], levels=30, cmap="YlGnBu", vmin = vmin, vmax = vmax)
ax[1].set_xlabel(r"$\mathbf{CO}_{\mathbf{2}}$ mole fraction", fontsize = 14, fontweight="bold")
ax[1].set_ylabel("Pressure (bar)", fontsize = 14, fontweight="bold")
ax[1].set_title(r"$\mathrm{CO}_{2}$ Prediction Residuals for P-X Space", fontsize = 14)
for tick in ax[1].get_xticklabels() + ax[1].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")

cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.75])
cbar = fig.colorbar(contour_co, cax = cbar_ax, label="Absolute Residual (mg/g)", shrink=0.9)
cbar.set_label("Absolute Residual (mg/g)", fontsize=14)
cbar.ax.tick_params(labelsize=12)
for i, axes in enumerate(ax.flat):
    axes.text(-0.15,1.06,panel_labels[i],transform=axes.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False)
plt.tight_layout()
plt.savefig('Residual_px.png', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.savefig('Residual_px.pdf', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.show()